In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from xgboost import XGBClassifier

In [3]:
df = pd.read_csv("../../data/product_type_2.csv", header=[0,1])
df

process                                                     \
           id product_type shot velocity_1 velocity_2 velocity_3   
0     4207011            2   11      0.156      0.166      0.192   
1     4208012            2   12      0.157      0.166      0.204   
2     4209013            2   13      0.156      0.170      0.204   
3     4210014            2   14      0.154      0.170      0.202   
4     4211015            2   15      0.146      0.160      0.198   
...       ...          ...  ...        ...        ...        ...   
1959  7525657            2  657      0.144      0.173      0.200   
1960  7527658            2  658      0.144      0.173      0.200   
1961  7529659            2  659      0.150      0.166      0.210   
1962  7531660            2  660      0.144      0.174      0.206   
1963  7533661            2  661      0.147      0.174      0.204   

                                                                        ...  \
     high_velocity cylinder_pressure rapid_rise_time biscuit_thickness  ...   
0            2.723               265           0.012                20  ...   
1            2.730               264           0.014                19  ...   
2            2.715               265           0.012                18  ...   
3            2.717               264           0.011                20  ...   
4            2.684               264           0.012                20  ...   
...            ...               ...             ...               ...  ...   
1959         2.536               264           0.012                17  ...   
1960         2.536               264           0.012                17  ...   
1961         2.492               265           0.011                17  ...   
1962         2.514               264           0.011                16  ...   
1963         2.532               265           0.012                18  ...   

     defects                                                          \
     stain_2 dent_2 deformation_2 contamination_2 impurity_2 crack_2   
0          0      0             0               0          0       0   
1          0      0             0               0          0       0   
2          0      0             0               0          0       0   
3          0      0             0               0          0       0   
4          0      0             0               0          0       0   
...      ...    ...           ...             ...        ...     ...   
1959       0      0             0               0          0       0   
1960       0      0             0               0          0       0   
1961       0      0             0               0          0       0   
1962       0      0             0               0          0       0   
1963       0      0             0               0          0       0   

                                          defect_flag  
     scratch_2 buring_mark_2 inclusions_2   is_defect  
0            0             0            0           0  
1            0             0            0           0  
2            0             0            0           0  
3            0             0            0           0  
4            0             0            0           0  
...        ...           ...          ...         ...  
1959         0             0            0           0  
1960         0             0            0           0  
1961         0             0            0           0  
1962         0             0            0           1  
1963         0             0            0           1  

[1964 rows x 58 columns]

In [4]:
### 1. 불필요한 id, product_type, shot 컬럼 제거
df = df.drop(columns=[('process', 'id'), ('process', 'shot'), ('process', 'product_type')])

In [5]:
### 2. 값이 전부 0인 defects 컬럼 drop
# Defects 하위 컬럼들
defect_cols = df['defects'].columns

# 값이 전부 0인 Defects 컬럼 찾기
zero_defects = [
    col for col in defect_cols
    if df[('defects', col)].sum() == 0
]

# 결과 출력
print("삭제될 defects 컬럼:")
print(zero_defects)

print("\n삭제될 컬럼 수:", len(zero_defects))
print("삭제 전 defects 컬럼 수:", len(defect_cols))

# 실제 삭제
df = df.drop(columns=[('defects', col) for col in zero_defects])

print("삭제 후 defects 컬럼 수:", len(df['defects'].columns))

삭제될 defects 컬럼:
['exfoliation_1', 'deformation_1', 'inclusions_1', 'bubble_2', 'exfoliation_2', 'stain_2', 'deformation_2', 'scratch_2', 'buring_mark_2']

삭제될 컬럼 수: 9
삭제 전 defects 컬럼 수: 26
삭제 후 defects 컬럼 수: 17


In [6]:
df['defects'].sum().sort_values(ascending=False)

short_shot_1       176
blow_hole_1        112
stain_1             93
blow_hole_2         79
short_shot_2        48
contamination_2      8
buring_mark_1        5
impurity_2           5
dent_1               4
contamination_1      4
dent_2               4
impurity_1           2
scratch_1            2
crack_2              2
bubble_1             1
crack_1              1
inclusions_2         1
dtype: int64

In [7]:
# 불량 범주 정의 (suffix _1, _2 자동 처리)
defect_groups = {
    "표면": [
        "stain_1",
        "dent_1",
        "dent_2",
        "scratch_1",
        "buring_mark_1"
    ],

    "구조": [
        "short_shot_1",
        "short_shot_2",
        "blow_hole_1",
        "blow_hole_2",
        "bubble_1",
        "crack_1",
        "crack_2"
    ],

    "이물질": [
        "contamination_1",
        "contamination_2",
        "impurity_1",
        "impurity_2",
        "inclusions_2"
    ]
}

In [9]:
import numpy as np
import pandas as pd

defects = df['defects'].copy()  # columns: Short_Shot_1, Blow_Hole_2, ...

# 각 범주별로 해당하는 defect 컬럼들을 찾아서 "하나라도 1이면 1"로 만들기
y_group = pd.DataFrame(index=df.index)

for group_name, base_names in defect_groups.items():
    # base_names 중 하나로 시작하는 컬럼들 찾기 (예: "Short_Shot" -> "Short_Shot_1", "Short_Shot_2")
    cols = [c for c in defects.columns if any(str(c).startswith(b) for b in base_names)]
    
    if len(cols) == 0:
        # 해당 범주에 매칭되는 컬럼이 없으면 0으로
        y_group[group_name] = 0
    else:
        y_group[group_name] = (defects[cols].fillna(0).astype(int).sum(axis=1) > 0).astype(int)

print(y_group.sum().sort_values(ascending=False))   # 범주별 발생 건수 확인
print(y_group.mean().mul(100))                      # 범주별 발생률(%) 확인

구조     395
표면     107
이물질     19
dtype: int64
표면      5.448065
구조     20.112016
이물질     0.967413
dtype: float64


In [10]:
# 입력 변수
X = df['process'].copy()

# 혹시 결측이 있으면 일단 중앙값으로 채우기
X = X.fillna(X.median())

# Stage 1 타겟: 하나라도 불량이면 1
y_binary = (y_group.sum(axis=1) > 0).astype(int)

print("정상/불량 분포")
print(y_binary.value_counts())
print(y_binary.value_counts(normalize=True))

정상/불량 분포
0    1467
1     497
Name: count, dtype: int64
0    0.746945
1    0.253055
Name: proportion, dtype: float64


In [11]:
X_train, X_test, y_train_bin, y_test_bin, y_train_group, y_test_group = train_test_split(
    X, y_binary, y_group,
    test_size=0.2,
    random_state=42,
    stratify=y_binary
)

print(X_train.shape, X_test.shape)

(1571, 14) (393, 14)


In [12]:
# SMOTE 적용
smote_stage1 = SMOTE(random_state=42)
X_train_sm1, y_train_bin_sm = smote_stage1.fit_resample(X_train, y_train_bin)

print("SMOTE 후 Stage1 분포")
print(pd.Series(y_train_bin_sm).value_counts())

# Stage 1 모델
stage1_model = XGBClassifier(
    tree_method="hist",
    n_estimators=200,
    max_depth=5,
    learning_rate=0.08,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

stage1_model.fit(X_train_sm1, y_train_bin_sm)

SMOTE 후 Stage1 분포
1    1173
0    1173
Name: count, dtype: int64


,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes f

In [14]:
# 불량 확률
stage1_proba = stage1_model.predict_proba(X_test)[:, 1]

# threshold 조정
threshold = 0.25
stage1_pred = (stage1_proba >= threshold).astype(int)

print("=== Stage 1: 불량 여부 분류 ===")
print(classification_report(y_test_bin, stage1_pred, zero_division=0))

print("Confusion Matrix")
print(confusion_matrix(y_test_bin, stage1_pred))

=== Stage 1: 불량 여부 분류 ===
              precision    recall  f1-score   support

           0       0.82      0.57      0.67       294
           1       0.33      0.64      0.44        99

    accuracy                           0.59       393
   macro avg       0.58      0.60      0.55       393
weighted avg       0.70      0.59      0.61       393

Confusion Matrix
[[167 127]
 [ 36  63]]


In [16]:
# 단일 불량만 있는 행만 사용
single_defect_mask = (y_group.sum(axis=1) == 1)

# 대표 불량 유형
y_type = y_group.idxmax(axis=1)

# Stage 2 전체 후보
X_type = X[single_defect_mask]
y_type = y_type[single_defect_mask]

print("Stage 2 학습 후보 분포")
print(y_type.value_counts())
print(y_type.value_counts(normalize=True))

Stage 2 학습 후보 분포
구조     371
표면      83
이물질     19
Name: count, dtype: int64
구조     0.784355
표면     0.175476
이물질    0.040169
Name: proportion, dtype: float64


In [17]:
print("전체 불량 데이터 수:", (y_group.sum(axis=1) > 0).sum())
print("단일 불량 데이터 수:", single_defect_mask.sum())

전체 불량 데이터 수: 497
단일 불량 데이터 수: 473


In [20]:
# Stage 1 split 기준으로 Stage 2 train/test 구성
train_idx = X_train.index
test_idx = X_test.index

stage2_train_mask = single_defect_mask.loc[train_idx]
stage2_test_mask = single_defect_mask.loc[test_idx]

X_train_stage2 = X_train[single_defect_mask.loc[X_train.index]]
y_train_stage2 = y_type.loc[X_train_stage2.index]

X_test_stage2 = X_test[single_defect_mask.loc[X_test.index]]
y_test_stage2 = y_type.loc[X_test_stage2.index]

print("Stage 2 train shape:", X_train_stage2.shape)
print("Stage 2 test shape:", X_test_stage2.shape)
print("\nStage 2 train class 분포")
print(y_train_stage2.value_counts())

Stage 2 train shape: (377, 14)
Stage 2 test shape: (96, 14)

Stage 2 train class 분포
구조     293
표면      68
이물질     16
Name: count, dtype: int64


In [22]:
# 문자열 라벨 → 숫자 라벨
label_map = {'표면': 0, '구조': 1, '이물질': 2}
inv_label_map = {v: k for k, v in label_map.items()}

y_train_stage2_num = y_train_stage2.map(label_map)
y_test_stage2_num = y_test_stage2.map(label_map)

# SMOTE
from imblearn.over_sampling import SMOTE

smote_stage2 = SMOTE(random_state=42, k_neighbors=1)
X_train_sm2, y_train_stage2_sm = smote_stage2.fit_resample(X_train_stage2, y_train_stage2_num)

print("\nSMOTE 후 Stage2 분포")
print(pd.Series(y_train_stage2_sm).value_counts())

# XGBoost
from xgboost import XGBClassifier

stage2_model = XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    tree_method="hist",
    n_estimators=200,
    max_depth=5,
    learning_rate=0.08,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1
)

stage2_model.fit(X_train_sm2, y_train_stage2_sm)

# 예측
stage2_pred_num = stage2_model.predict(X_test_stage2)
stage2_pred = pd.Series(stage2_pred_num, index=X_test_stage2.index).map(inv_label_map)


SMOTE 후 Stage2 분포
1    293
0    293
2    293
Name: count, dtype: int64


In [23]:

print("=== Stage 2: 불량 종류 분류 ===")
print(classification_report(y_test_stage2, stage2_pred, zero_division=0))

=== Stage 2: 불량 종류 분류 ===
              precision    recall  f1-score   support

          구조       0.83      0.87      0.85        78
         이물질       0.29      0.67      0.40         3
          표면       0.00      0.00      0.00        15

    accuracy                           0.73        96
   macro avg       0.37      0.51      0.42        96
weighted avg       0.68      0.73      0.70        96



smote, xgboost

In [24]:
X_train_stage2.groupby(y_train_stage2).mean()

,velocity_1,velocity_2,velocity_3,high_velocity,cylinder_pressure,rapid_rise_time,biscuit_thickness,clamping_force,cycle_time,pressure_rise_time,casting_pressure,spray_time,spray_1_time,spray_2_time
구조,0.154191,0.168945,0.201935,2.524659,264.767918,0.011597,17.648464,373.061433,35.753925,0.036109,595.378840,11.857338,2.021843,2.116724
이물질,0.155500,0.168750,0.202750,2.705312,264.625000,0.011750,18.375000,358.062500,33.881250,0.041188,595.250000,10.056250,2.000000,2.700000
표면,0.154088,0.169015,0.200706,2.548735,264.867647,0.011779,17.676471,369.911765,35.833824,0.037221,595.661765,11.941176,2.029412,2.123529


In [25]:
# Stage 1 확률 / 예측
stage1_proba_test = stage1_model.predict_proba(X_test)[:, 1]
stage1_pred_test = (stage1_proba_test >= threshold).astype(int)

# 기본 결과 테이블
final_pred = pd.DataFrame(index=X_test.index)
final_pred["불량확률"] = stage1_proba_test
final_pred["불량여부_예측"] = stage1_pred_test
final_pred["예측_불량종류"] = "정상"

# Stage 1에서 불량으로 예측된 샘플만 Stage 2 분류
pred_defect_idx = final_pred.index[final_pred["불량여부_예측"] == 1]

if len(pred_defect_idx) > 0:
    stage2_pred_for_defects = stage2_model.predict(X_test.loc[pred_defect_idx])
    final_pred.loc[pred_defect_idx, "예측_불량종류"] = stage2_pred_for_defects

final_pred.head(20)

TypeError: Invalid value for dtype 'str'. Value should be a string or missing value (or array of those).

In [ ]:
actual_result = pd.DataFrame(index=X_test.index)
actual_result["실제_불량여부"] = y_test_bin.values
actual_result["실제_불량종류"] = "정상"

# 실제 단일 불량 범주인 경우만 실제 종류 표시
single_test_idx = y_test_group.index[(y_test_group.sum(axis=1) == 1)]
actual_result.loc[single_test_idx, "실제_불량종류"] = y_test_group.loc[single_test_idx].idxmax(axis=1)

result_compare = final_pred.join(actual_result)
result_compare.head(20)